
# Appendix A reconstructed from the report

**Pim van den Bos · Management Control for Financial Institutions · University of Twente**

This is a reconstruction of PDF Appendix A.1-A.13, not the unavailable original
notebook and not a byte-identical source recovery. Strings, indentation, operators
and wrapped lines have been repaired. Standalone functions and both combined
engines are retained step by step. See [code provenance](../docs/code_provenance.md).

Run from the repository root or `notebooks/` after installing the project and
explicitly acquiring the DNB workbook. The default is a **256-scenario demonstration**.
Set `PENSION_SCENARIOS=all` before opening the kernel to select the entire workbook;
`PENSION_WORKBOOK` accepts a local workbook path. No cells download data.
Outputs are deliberately empty: no published simulation evidence is fabricated.
The practical package avoids the intentional duplication in this reconstruction.


## A.1 Environment setup

The PDF installs `python-calamine numpy pandas scipy matplotlib` with a bare
`pip install` line. The portable equivalent is `python -m pip install -e ".[dev]"`
from the repository root, using `requirements-lock.txt` for the tested versions.
SciPy is unused by every listing and is not required. Select the installed environment
as your notebook kernel. Installation is kept outside this executable cell.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print("Kernel:", sys.version.split()[0])


## A.2 Loading the DNB scenario set and reconstructing the term structure

The numeric loading and curve formulas are transcribed directly. Added source fingerprint and shape checks use package helpers. Calamine opens the workbook once, and the NPZ cache is tied to the source bytes.

In [ ]:
# Step 1 -- Load DNB CP2022-P scenarios + reconstruct the term structure.
# First run reads the XLSX and writes an NPZ; later runs load a source-bound cache.
import os
import json
import numpy as np
import pandas as pd
from pension_risk.acquisition import KNOWN_SHA256, sha256_file
from pension_risk.data import validate_arrays

XLSX = Path(os.environ.get("PENSION_WORKBOOK", ROOT / "data/raw/cp2022-p-scenarioset-20k-2026q2.xlsx"))
fingerprint = sha256_file(XLSX)
if fingerprint != KNOWN_SHA256:
    raise ValueError("Workbook differs from the recorded 2026Q2 P-set")
CACHE = ROOT / "data/cache" / f"appendix-v1-{fingerprint}.npz"
SHEETS = ["4_Aandelenrendement", "6_Prijsinflatie_NL",
          "1_Toestandsvariabele_1", "2_Toestandsvariabele_2", "3_Toestandsvariabele_3",
          "7_Renteparameter_phi_N", "8_Renteparameter_Psi_N",
          "11_Renteparameter_phi_R_NL", "10_Renteparameter_Psi_R"]
KEYS = ["equity", "infl_nl", "s1", "s2", "s3", "phi_N", "Psi_N", "phi_R_NL", "Psi_R"]

def read_sheet(name):
    arr = pd.read_excel(book, sheet_name=name, header=None).to_numpy(dtype=float)
    arr = arr[~np.all(np.isnan(arr), axis=1)]
    arr = arr[:, ~np.all(np.isnan(arr), axis=0)]
    return arr

# Added source fingerprint/shape checks; original NPZ loading structure retained.
metadata = {"sha256": fingerprint, "schema": 1}
if CACHE.exists():
    with np.load(CACHE, allow_pickle=False) as z:
        if json.loads(str(z["metadata"])) != metadata:
            raise ValueError("Cache source/schema mismatch")
        arrays = {k: z[k] for k in KEYS}
else:
    with pd.ExcelFile(XLSX, engine="calamine") as book:
        if not set(SHEETS) <= set(book.sheet_names):
            raise ValueError("Missing required sheets")
        arrays = {k: read_sheet(s) for k, s in zip(KEYS, SHEETS)}
    validate_arrays(arrays)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(CACHE, **arrays, metadata=json.dumps(metadata))
validate_arrays(arrays)
selection = os.environ.get("PENSION_SCENARIOS", "256")
if selection != "all":
    limit = int(selection)
    if not 2 <= limit <= arrays["equity"].shape[0]:
        raise ValueError("PENSION_SCENARIOS must be 2..20000, or all")
    for key in ("equity", "infl_nl", "s1", "s2", "s3"):
        arrays[key] = arrays[key][:limit]
print("Selection:", selection, "(subset outputs are demonstrations)")

equity, infl_nl = arrays["equity"], arrays["infl_nl"]
s1, s2, s3 = arrays["s1"], arrays["s2"], arrays["s3"]
phi_N, Psi_N = arrays["phi_N"], arrays["Psi_N"]
phi_R_NL, Psi_R = arrays["phi_R_NL"], arrays["Psi_R"]
n_scen = equity.shape[0]

def curve(phi, Psi, t, maturities, scenarios):
    """y(n,s,t)=exp(-1/n * (phi[n,t] + sum_k S_k[s,t]*Psi[n,k]))-1."""
    idx = np.atleast_1d(maturities) - 1
    n = np.atleast_1d(maturities).astype(float)
    states = np.stack([s1[scenarios, t], s2[scenarios, t], s3[scenarios, t]], axis=1)
    return np.exp((-1.0 / n)[None, :] *
                  (phi[idx, t][None, :] + states @ Psi[idx, :].T)) - 1.0

def nominal_curve(t, maturities, scenarios):
    return curve(phi_N, Psi_N, t, maturities, scenarios)

def real_curve_nl(t, maturities, scenarios):
    return curve(phi_R_NL, Psi_R, t, maturities, scenarios)

print(equity.shape, infl_nl.shape)
print(np.round(nominal_curve(0, np.arange(1, 31), [0])[0] * 100, 2))


## A.3 Baseline cohort engine (no damping)



In [ ]:
ENTRY_AGE = 25
RETIREMENT_AGE = 68
END_AGE = 95  # fixed horizon, no mortality in the main engine
CONTRIBUTION = 1.0  # flat nominal contribution, before return
PROJECTION = 0.03
n_scen = equity.shape[0]
acc_years = RETIREMENT_AGE - ENTRY_AGE
pay_years = END_AGE - RETIREMENT_AGE

def run_baseline(scenarios=None):
    """Baseline SPR: 100% risky exposure, full pass-through, no damping.
    Returns nominal and real benefits, each shape (n_scen, pay_years).
    """
    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for k in range(acc_years):
        capital = (capital + CONTRIBUTION) * (1.0 + equity[s, k])
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + PROJECTION) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor
    for k in range(1, pay_years):
        t = acc_years + k - 1
        benefit[:, k] = benefit[:, k - 1] * (1.0 + equity[s, t]) / (1.0 + PROJECTION)
    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_idx = np.clip(acc_years + np.arange(pay_years) - 1, 0, equity.shape[1] - 1)
    real_benefit = benefit / price_level[:, pay_idx]
    return benefit, real_benefit

nominal, real = run_baseline()
print("benefit shape:", real.shape)
print("median real benefit, first payout year:", np.median(real[:, 0]).round(3))
print("5th/50th/95th pct, first year:", np.percentile(real[:, 0], [5, 50, 95]).round(3))
print("median year-on-year benefit volatility:",
      np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1)).round(3))


## A.4 Standalone spreading mechanism

The PDF prose says “next N years”; the executable ring buffer includes the current update. Its code timing is preserved. The original `first_benefit` temporary is inlined into `benefit[:, 0]`.

In [ ]:
SPREAD_YEARS = 5

def run_with_spreading(spread_years=SPREAD_YEARS, scenarios=None):
    """Baseline payout with equal, interest-free slices of each return shock.
    The current shock's first slice participates in the current update.
    """

    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for t in range(acc_years):
        capital = (capital + CONTRIBUTION) * (1.0 + equity[s, t])
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + PROJECTION) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor

    M = max(int(spread_years), 1)
    pending = np.zeros((N, M))
    for k in range(1, pay_years):
        t = acc_years + k - 1
        raw_shock = (1.0 + equity[s, t]) / (1.0 + PROJECTION) - 1.0
        pending[:, k % M] = raw_shock / M
        applied = pending.sum(axis=1)
        benefit[:, k] = benefit[:, k - 1] * (1.0 + applied)

    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc_years + np.arange(pay_years)
    real_benefit = benefit / price_level[:, pay_data_years - 1]
    return benefit, real_benefit

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

_, real_base = run_baseline()
_, real_spread = run_with_spreading(5)
print("yoy benefit volatility baseline:", round(yoy_vol(real_base), 3))
print("yoy benefit volatility + spreading N=5:", round(yoy_vol(real_spread), 3))


## A.5 Standalone solidarity reserve (single-cohort buffer)



In [ ]:
RESERVE_FILL_RATE = 0.25
RESERVE_CAP = 2.0

def run_with_reserve(fill_rate=RESERVE_FILL_RATE, cap=RESERVE_CAP, scenarios=None):
    """Skim positive benefit surplus; top up cuts while this participant has funds.
    Initially empty; cap in units of the previous annual benefit.
    Intergenerational sharing would require a separate collective layer.
    """

    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for t in range(acc_years):
        capital = (capital + CONTRIBUTION) * (1.0 + equity[s, t])
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + PROJECTION) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor

    reserve = np.zeros(N)
    for k in range(1, pay_years):
        t = acc_years + k - 1
        raw = benefit[:, k - 1] * (1.0 + equity[s, t]) / (1.0 + PROJECTION)
        target = benefit[:, k - 1]
        shortfall = np.maximum(target - raw, 0.0)
        topup = np.minimum(shortfall, reserve)
        b = raw + topup
        reserve -= topup
        surplus = np.maximum(raw - target, 0.0)
        room = np.maximum(cap * benefit[:, k - 1] - reserve, 0.0)
        contrib = np.minimum(fill_rate * surplus, room)
        b -= contrib
        reserve += contrib
        benefit[:, k] = b

    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc_years + np.arange(pay_years)
    real_benefit = benefit / price_level[:, pay_data_years - 1]
    return benefit, real_benefit

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

_, real_base = run_baseline()
_, real_reserve = run_with_reserve()
print("yoy volatility baseline:", round(yoy_vol(real_base), 3))
print("yoy volatility + reserve:", round(yoy_vol(real_reserve), 3))
print("median real benefit baseline/reserve:", round(np.median(real_base[:, 0]), 3),
      "/", round(np.median(real_reserve[:, 0]), 3))


## A.6 Standalone protection return (interest-rate hedge)



In [ ]:
def hedge_ratio(age):
    """Linear ramp from zero at age 45 to one at retirement, then one."""
    return float(np.clip((age - 45) / (RETIREMENT_AGE - 45), 0.0, 1.0))

def liability_duration(age):
    """Half the remaining fixed life horizon, floored at one."""
    years_left = max(END_AGE - age, 1)
    return max(years_left / 2.0, 1.0)

def discount(t, maturity, scenarios):
    """Nominal zero-coupon P_t(maturity) = (1+y)^(-maturity)."""
    if maturity <= 0:
        return np.ones(len(np.atleast_1d(scenarios)))
    y = nominal_curve(t, [maturity], scenarios)[:, 0]
    return (1.0 + y) ** (-maturity)

def protection_return(t, scenarios, duration):
    """One-year bond holding return P_{t+1}(d-1)/P_t(d)-1."""
    d = int(round(duration))
    return discount(t + 1, d - 1, scenarios) / discount(t, d, scenarios) - 1.0

def run_with_hedge(scenarios=None):
    """Allocated return = h*protection + (1-h)*equity."""
    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for t in range(acc_years):
        age = ENTRY_AGE + t
        h = hedge_ratio(age)
        eq = equity[s, t]
        if h > 0:
            prot = protection_return(t, s, liability_duration(age))
            allocated = h * prot + (1.0 - h) * eq
        else:
            allocated = eq
        capital = (capital + CONTRIBUTION) * (1.0 + allocated)
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + PROJECTION) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor
    for k in range(1, pay_years):
        t = acc_years + k - 1
        age = RETIREMENT_AGE + k - 1
        h = hedge_ratio(age)
        prot = protection_return(t, s, liability_duration(age))
        allocated = h * prot + (1.0 - h) * equity[s, t]
        benefit[:, k] = benefit[:, k - 1] * (1.0 + allocated) / (1.0 + PROJECTION)

    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc_years + np.arange(pay_years)
    real_benefit = benefit / price_level[:, pay_data_years - 1]
    return benefit, real_benefit

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

_, real_base = run_baseline()
_, real_hedge = run_with_hedge()
print("yoy volatility baseline:", round(yoy_vol(real_base), 3))
print("yoy volatility + hedge:", round(yoy_vol(real_hedge), 3))


## A.7 Standalone age-dependent excess return allocation (lifecycle)



In [ ]:
def exposure(age):
    """Full risk to age 35, then declining to 0.2 at END_AGE (95)."""
    return float(np.clip(1.0 - 0.8 * (age - 35) / (END_AGE - 35), 0.2, 1.0))

def risk_free_1y(t, scenarios):
    """Reconstructed one-year nominal risk-free rate at data time t."""
    return nominal_curve(t, [1], scenarios)[:, 0]

def run_with_lifecycle(scenarios=None):
    """Allocated return = w(age)*equity + (1-w(age))*risk_free."""
    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for t in range(acc_years):
        age = ENTRY_AGE + t
        w = exposure(age)
        rf = risk_free_1y(t, s)
        allocated = w * equity[s, t] + (1.0 - w) * rf
        capital = (capital + CONTRIBUTION) * (1.0 + allocated)
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + PROJECTION) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor
    for k in range(1, pay_years):
        t = acc_years + k - 1
        age = RETIREMENT_AGE + k - 1
        w = exposure(age)
        rf = risk_free_1y(t, s)
        allocated = w * equity[s, t] + (1.0 - w) * rf
        benefit[:, k] = benefit[:, k - 1] * (1.0 + allocated) / (1.0 + PROJECTION)

    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc_years + np.arange(pay_years)
    real_benefit = benefit / price_level[:, pay_data_years - 1]
    return benefit, real_benefit

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

_, real_base = run_baseline()
_, real_lifecycle = run_with_lifecycle()
print("yoy volatility baseline:", round(yoy_vol(real_base), 3))
print("yoy volatility + lifecycle:", round(yoy_vol(real_lifecycle), 3))
print("median real benefit baseline/lifecycle:", round(np.median(real_base[:, 0]), 3),
      "/", round(np.median(real_lifecycle[:, 0]), 3))


## A.8 Projection return sensitivity (excluded from damping set)



In [ ]:
def run_with_projection(projection, scenarios=None):
    """Baseline mechanics; projection sets both annuity conversion and updates."""

    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for t in range(acc_years):
        capital = (capital + CONTRIBUTION) * (1.0 + equity[s, t])
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + projection) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor

    for k in range(1, pay_years):
        t = acc_years + k - 1
        benefit[:, k] = benefit[:, k - 1] * (1.0 + equity[s, t]) / (1.0 + projection)

    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc_years + np.arange(pay_years)
    real_benefit = benefit / price_level[:, pay_data_years - 1]
    return benefit, real_benefit

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

for proj in (0.025, 0.03, 0.04):
    _, real = run_with_projection(proj)
    start = np.median(real[:, 0])
    end = np.median(real[:, -1])
    print(f"projection={proj:.3f} median start={start:6.3f} median end={end:6.3f} "
          f"drift={end/start-1:+.1%} yoy_vol={yoy_vol(real):.3f}")


## A.9 Collective payout / longevity illustration (excluded from damping set)



In [ ]:
def mortality_rate(age):
    """Illustrative q(age), roughly doubling every nine years; not an AG table."""
    return float(np.clip(0.005 * 2 ** ((age - 65) / 9.0), 0.0, 1.0))

def run_with_longevity(scenarios=None):
    """Deterministic cohort survival and per-survivor mortality credits.
    Includes survival-weighted conversion and 1/(1-q) payout scaling as printed.
    """
    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    capital = np.zeros(N)
    for t in range(acc_years):
        capital = (capital + CONTRIBUTION) * (1.0 + equity[s, t])
    surv = 1.0
    survival = []
    for k in range(pay_years):
        survival.append(surv)
        surv *= (1.0 - mortality_rate(RETIREMENT_AGE + k))
    survival = np.array(survival)
    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum(survival * (1.0 + PROJECTION) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor
    for k in range(1, pay_years):
        t = acc_years + k - 1
        q = mortality_rate(RETIREMENT_AGE + k - 1)
        mortality_return = 1.0 / (1.0 - q)
        allocated = (1.0 + equity[s, t]) * mortality_return
        benefit[:, k] = benefit[:, k - 1] * allocated / (1.0 + PROJECTION)

    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc_years + np.arange(pay_years)
    real_benefit = benefit / price_level[:, pay_data_years - 1]
    return benefit, real_benefit

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

_, real_base = run_baseline()
_, real_long = run_with_longevity()
print("yoy volatility baseline/longevity:", round(yoy_vol(real_base), 3), "/", round(yoy_vol(real_long), 3))
print("median real benefit baseline/longevity:", round(np.median(real_base[:, 0]), 3),
      "/", round(np.median(real_long[:, 0]), 3))


## A.10 Combined engine with four on/off switches



In [ ]:
def hedge_ratio(age):
    """Linear ramp from zero at age 45 to one at retirement, then one."""
    return float(np.clip((age - 45) / (RETIREMENT_AGE - 45), 0.0, 1.0))

def liability_duration(age):
    """Half the remaining fixed life horizon, floored at one."""
    years_left = max(END_AGE - age, 1)
    return max(years_left / 2.0, 1.0)

def discount(t, maturity, scenarios):
    """Nominal zero-coupon P_t(maturity) = (1+y)^(-maturity)."""
    if maturity <= 0:
        return np.ones(len(np.atleast_1d(scenarios)))
    y = nominal_curve(t, [maturity], scenarios)[:, 0]
    return (1.0 + y) ** (-maturity)

def protection_return(t, scenarios, duration):
    """One-year bond holding return P_{t+1}(d-1)/P_t(d)-1."""
    d = int(round(duration))
    return discount(t + 1, d - 1, scenarios) / discount(t, d, scenarios) - 1.0

def exposure(age):
    """Full risk to age 35, then declining to 0.2 at END_AGE (95)."""
    return float(np.clip(1.0 - 0.8 * (age - 35) / (END_AGE - 35), 0.2, 1.0))

def risk_free_1y(t, scenarios):
    """Reconstructed one-year nominal risk-free rate at data time t."""
    return nominal_curve(t, [1], scenarios)[:, 0]

def run(hedge=False, lifecycle=False, spreading=False, reserve=False,
        spread_years=5, fill_rate=0.25, cap=2.0, projection=PROJECTION,
        entry_age=ENTRY_AGE, scenarios=None):
    """A.10: zero initial capital; allocation, raw shock, spreading, reserve.
    Projection and longevity extensions are separate from the four switches.
    """
    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    M = max(int(spread_years), 1)
    acc = RETIREMENT_AGE - entry_age

    def allocate(t, age):
        eq = equity[s, t]
        w = exposure(age) if lifecycle else 1.0
        h = hedge_ratio(age) if hedge else 0.0
        risky = w * eq + (1.0 - w) * (risk_free_1y(t, s) if w < 1.0 else 0.0)
        if h > 0.0:
            return (h * protection_return(t, s, liability_duration(age))
                    + (1.0 - h) * risky)
        return risky

    capital = np.zeros(N)
    for t in range(acc):
        capital = (capital + CONTRIBUTION) * (1.0 + allocate(t, entry_age + t))

    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + projection) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor
    pending = np.zeros((N, M))
    res = np.zeros(N)
    for k in range(1, pay_years):
        t = acc + k - 1
        allocated = allocate(t, RETIREMENT_AGE + k - 1)
        raw_shock = (1.0 + allocated) / (1.0 + projection) - 1.0
        if spreading:
            pending[:, k % M] = raw_shock / M
            applied = pending.sum(axis=1)
        else:
            applied = raw_shock
        b = benefit[:, k - 1] * (1.0 + applied)
        if reserve:
            target = benefit[:, k - 1]
            topup = np.minimum(np.maximum(target - b, 0.0), res)
            b += topup
            res -= topup
            room = np.maximum(cap * benefit[:, k - 1] - res, 0.0)
            contrib = np.minimum(fill_rate * np.maximum(b - target, 0.0), room)
            b -= contrib
            res += contrib
        benefit[:, k] = b
    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc + np.arange(pay_years)
    return benefit, benefit / price_level[:, pay_data_years - 1]

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

nb, rb = run_baseline()
print("all-off == baseline:", np.allclose(nb, run(False, False, False, False)[0]))
print("baseline yoy vol:", round(yoy_vol(rb), 3))
for name, kw in [("hedge", dict(hedge=True)), ("lifecycle", dict(lifecycle=True)),
                 ("spreading", dict(spreading=True)), ("reserve", dict(reserve=True))]:
    print(f"+{name:9s}:", round(yoy_vol(run(**kw)[1]), 3))
print("all four:", round(yoy_vol(run(True, True, True, True)[1]), 3))


## A.11 Deterministic starting capital, cohort grid and risk metrics

The PDF comment says YoY is the same for all ages; the function computes it for each cohort. Describe it as broadly similar, without forcing equality. Historical capital uses the fixed global 3% assumption even if the forward `projection` argument differs.

In [ ]:
GRID = [25, 35, 45, 55, 67]

def accumulated_to(age):
    """Contributions since age 25 grown at fixed PROJECTION; no past shocks."""
    cap = 0.0
    for _ in range(age - ENTRY_AGE):
        cap = (cap + CONTRIBUTION) * (1.0 + PROJECTION)
    return cap

def run_grid(start_age, hedge=False, lifecycle=False, spreading=False, reserve=False,
             spread_years=5, fill_rate=0.25, cap=2.0, projection=PROJECTION, scenarios=None):
    """A.11: current-age engine seeded with accumulated_to(start_age)."""
    s = np.arange(n_scen) if scenarios is None else np.atleast_1d(scenarios)
    N = len(s)
    M = max(int(spread_years), 1)
    acc = max(RETIREMENT_AGE - start_age, 0)

    def allocate(t, age):
        eq = equity[s, t]
        w = exposure(age) if lifecycle else 1.0
        h = hedge_ratio(age) if hedge else 0.0
        risky = w * eq + (1.0 - w) * (risk_free_1y(t, s) if w < 1.0 else 0.0)
        if h > 0.0:
            return (h * protection_return(t, s, liability_duration(age))
                    + (1.0 - h) * risky)
        return risky

    capital = np.full(N, accumulated_to(start_age))
    for t in range(acc):
        capital = (capital + CONTRIBUTION) * (1.0 + allocate(t, start_age + t))

    horizon = np.arange(1, pay_years + 1)
    annuity_factor = np.sum((1.0 + projection) ** (-horizon))
    benefit = np.empty((N, pay_years))
    benefit[:, 0] = capital / annuity_factor
    pending = np.zeros((N, M))
    res = np.zeros(N)
    for k in range(1, pay_years):
        t = acc + k - 1
        allocated = allocate(t, RETIREMENT_AGE + k - 1)
        raw_shock = (1.0 + allocated) / (1.0 + projection) - 1.0
        if spreading:
            pending[:, k % M] = raw_shock / M
            applied = pending.sum(axis=1)
        else:
            applied = raw_shock
        b = benefit[:, k - 1] * (1.0 + applied)
        if reserve:
            target = benefit[:, k - 1]
            topup = np.minimum(np.maximum(target - b, 0.0), res)
            b += topup
            res -= topup
            room = np.maximum(cap * benefit[:, k - 1] - res, 0.0)
            contrib = np.minimum(fill_rate * np.maximum(b - target, 0.0), room)
            b -= contrib
            res += contrib
        benefit[:, k] = b
    price_level = np.cumprod(1.0 + infl_nl[s], axis=1)
    pay_data_years = acc + np.arange(pay_years)
    return benefit, benefit / price_level[:, pay_data_years - 1]

def yoy_vol(real):
    return np.median(np.std(np.diff(real, axis=1) / real[:, :-1], axis=1))

def level_cov(real):
    return np.median(np.std(real, axis=0) / np.mean(real, axis=0))

print(f"{'age':>4} {'baseline CoV':>12} {'all four CoV':>12} {'reduction':>10}")
for a in GRID:
    cb = level_cov(run_grid(a)[1])
    cd = level_cov(run_grid(a, hedge=True, lifecycle=True, spreading=True, reserve=True)[1])
    print(f"{a:>4} {cb:>12.3f} {cd:>12.3f} {1-cd/cb:>9.1%}")


## A.12 Full-factorial Shapley decomposition for both risk metrics

This cell retains the original repeated simulations for traceability. For practical full runs, the package caches each coalition summary once and uses it for both metrics and every table.

In [ ]:
from itertools import combinations
from math import factorial

MECHS = ["spreading", "reserve", "hedge", "lifecycle"]

def metric_value(age, active, metric):
    """Value of metric for the current cohort and active mechanism coalition."""
    flags = {m: (m in active) for m in MECHS}
    return metric(run_grid(age, **flags)[1])

def shapley_for_age(age, metric):
    """v(S) = risk(baseline) - risk(S); marginal risk reduction."""
    base = metric_value(age, frozenset(), metric)
    v = {frozenset(S): base - metric_value(age, frozenset(S), metric)
         for r in range(len(MECHS) + 1) for S in combinations(MECHS, r)}
    n = len(MECHS)
    phi = {m: 0.0 for m in MECHS}
    for m in MECHS:
        for r in range(len(MECHS)):
            for S in combinations([x for x in MECHS if x != m], r):
                S = frozenset(S)
                w = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
                phi[m] += w * (v[S | {m}] - v[S])
    return phi, v[frozenset(MECHS)]

def shapley_table(metric):
    rows = []
    for age in GRID:
        phi, total = shapley_for_age(age, metric)
        row = {"Age": age, "Total reduction": total}
        row.update({f"{m}_value": phi[m] for m in MECHS})
        row.update({f"{m}_share": 100 * phi[m] / total if total else 0.0 for m in MECHS})
        rows.append(row)
    return pd.DataFrame(rows)

shapley_df = shapley_table(level_cov)
shapley_df_yoy = shapley_table(yoy_vol)
val_cols = [f"{m}_value" for m in MECHS]
assert np.allclose(shapley_df[val_cols].sum(axis=1), shapley_df["Total reduction"])
assert np.allclose(shapley_df_yoy[val_cols].sum(axis=1), shapley_df_yoy["Total reduction"])
table5 = shapley_df.copy()
table6 = shapley_df[["Age", "spreading_share", "reserve_share", "hedge_share", "lifecycle_share"]].copy()
table6.columns = ["Age", "Spreading %", "Reserve %", "Protection Return %", "Lifecycle %"]
print(table6.round(1).to_string(index=False))


## A.13 Result-table generation (appendix numbering)

Appendix Tables 1-7 are generated calculations; they are not the four published tables in Section 6. See `results/reported/README.md` for the mapping. To save new estimates use the CLI, which writes to `results/generated/` with provenance.

In [ ]:
from IPython.display import display

FULL = dict(spreading=True, reserve=True, hedge=True, lifecycle=True)

# Appendix Table 1 -- baseline descriptives, all cohorts.
table1 = pd.DataFrame([{
    "Age": age, "Mean": np.mean(p), "Std": np.std(p), "CoV": level_cov(p),
    "P5": np.percentile(p, 5), "P95": np.percentile(p, 95)
} for age in GRID for p in [run_grid(age)[1]]]).round(4)
display(table1)

# Appendix Table 2 -- standalone mechanism effects at age 45.
base45 = level_cov(run_grid(45)[1])
mechanisms = {"Spreading": dict(spreading=True), "Reserve": dict(reserve=True),
              "Protection return": dict(hedge=True), "Lifecycle": dict(lifecycle=True)}
table2 = pd.DataFrame([{
    "Mechanism": name, "Mean": np.mean(p), "CoV": level_cov(p),
    "Delta CoV": base45 - level_cov(p), "Tail P95": np.percentile(p, 95)
} for name, st in mechanisms.items() for p in [run_grid(45, **st)[1]]]
).sort_values("Delta CoV", ascending=False).round(4)
display(table2)

# Appendix Table 3 -- sequential addition; attribution depends on this order.
steps = [("None", {}), ("Spreading", dict(spreading=True)),
         ("+Reserve", dict(spreading=True, reserve=True)),
         ("+Hedge", dict(spreading=True, reserve=True, hedge=True)), ("+Lifecycle", FULL)]
table3 = pd.DataFrame([{
    "Step": name, "CoV": level_cov(p), "Reduction": base45 - level_cov(p),
    "% Reduction": 100 * (base45 - level_cov(p)) / base45
} for name, st in steps for p in [run_grid(45, **st)[1]]]).round(3)
display(table3)

# Appendix Table 4 -- baseline vs full model for all cohorts.
table4 = pd.DataFrame([{
    "Age": age, "Baseline CoV": level_cov(run_grid(age)[1]),
    "Full CoV": level_cov(run_grid(age, **FULL)[1]),
    "Abs reduction": level_cov(run_grid(age)[1]) - level_cov(run_grid(age, **FULL)[1]),
    "% reduction": 100 * (1 - level_cov(run_grid(age, **FULL)[1]) / level_cov(run_grid(age)[1]))
} for age in GRID]).round(3)
display(table4)

# Appendix Tables 5/6 -- Shapley values and shares for level CoV.
display(shapley_df.round(4))
table6 = shapley_df[["Age", "spreading_share", "reserve_share", "hedge_share", "lifecycle_share"]].copy()
table6.columns = ["Age", "Spreading %", "Reserve %", "Protection Return %", "Lifecycle %"]
display(table6.round(1))

# Appendix Table 7 -- Shapley shares for YoY volatility.
table7 = shapley_df_yoy[["Age", "spreading_share", "reserve_share", "hedge_share", "lifecycle_share"]].copy()
table7.columns = ["Age", "Spreading %", "Reserve %", "Protection Return %", "Lifecycle %"]
display(table7.round(1))
